# Attention Is All You Need - Exploratory Analysis
**ArXivist-generated visualization notebook**

This notebook allows you to explore the internal representations and attention maps of the Transformer model. 

> **Note**: You must provide a trained checkpoint to fully utilize this notebook. If you haven't trained a model yet, run the primary reproduction notebook or `train.py` first, or download a pre-trained checkpoint.

In [ ]:
import sys, torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Note: Update these paths and configs based on your training run
checkpoint_path = "../checkpoints/best.pt"
model_config = {
    'N': 6,
    'd_model': 512,
    'd_ff': 2048,
    'h': 8,
    'd_k': 64,
    'd_v': 64,
    'P_drop': 0.1,
    'src_vocab_size': 37000,
    'tgt_vocab_size': 37000
}

try:
    from src.transformer.models.transformer import Transformer
    model = Transformer(**model_config).to(device)
    if torch.cuda.is_available():
        model.load_state_dict(torch.load(checkpoint_path))
    else:
        model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
    model.eval()
    print("Model loaded successfully.")
except FileNotFoundError:
    print(f"Checkpoint not found at {checkpoint_path}. Instantiating an untrained model for demonstration.")
    model = Transformer(**model_config).to(device)
    model.eval()
except Exception as e:
    print(f"Error loading model: {e}")

## Visualization 1: Self-Attention Maps
The Transformer relies heavily on self-attention. We can visualize the attention weights between different tokens in the input sequence. The self-attention mechanism is defined as:
$$ \text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V $$

In [ ]:
def plot_attention_map(attention_matrix, src_tokens, tgt_tokens=None, title="Attention Map"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(attention_matrix, xticklabels=src_tokens, yticklabels=tgt_tokens or src_tokens, 
                cmap="viridis", vmin=0.0, vmax=1.0)
    plt.title(title)
    plt.xlabel("Key Tokens")
    plt.ylabel("Query Tokens")
    plt.show()

# Mocking some attention weights for demonstration
try:
    seq_len = 10
    mock_tokens = [f"token_{i}" for i in range(seq_len)]
    # Simulate a single head's attention map for a 10-token sequence
    mock_attention = torch.softmax(torch.randn(seq_len, seq_len), dim=-1).numpy()
    
    plot_attention_map(mock_attention, mock_tokens, title="Mock Encoder Self-Attention (Head 1)")
except Exception as e:
    print(f"Error plotting attention: {e}")

## Visualization 2: Positional Encodings
The positional encodings are fixed sinusoidal waves added to the embeddings. We can visualize how the dimensions oscillate at different frequencies.

In [ ]:
try:
    from src.transformer.models.layers import PositionalEncoding
    d_model = 512
    max_len = 100
    pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_len)
    
    # Extract the pre-computed positional encodings
    pe = pos_encoder.pe.squeeze(0).numpy() # shape: [100, 512]
    
    plt.figure(figsize=(12, 6))
    plt.pcolormesh(pe, cmap='RdBu', vmin=-1.0, vmax=1.0)
    plt.title("Positional Encoding Matrix")
    plt.xlabel("Embedding Dimension")
    plt.ylabel("Sequence Position")
    plt.colorbar()
    plt.show()
except Exception as e:
    print(f"Error plotting positional encoding: {e}")

## Visualization 3: Multi-Head Comparison
Different attention heads often learn to focus on different aspects of the sequence (e.g., one head for local syntax, another for long-range dependencies). 
We can use `ipywidgets` to interactively switch between heads.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

try:
    num_heads = model_config['h']
    seq_len = 12
    mock_tokens_2 = [f"tok_{i}" for i in range(seq_len)]
    
    # Mocking attention matrices for 8 heads
    mock_multi_head_attention = [torch.softmax(torch.randn(seq_len, seq_len), dim=-1).numpy() for _ in range(num_heads)]
    
    def view_head(head_index):
        print(f"Visualizing Head {head_index}")
        plot_attention_map(mock_multi_head_attention[head_index], mock_tokens_2, title=f"Attention - Head {head_index}")
    
    head_slider = widgets.IntSlider(min=0, max=num_heads-1, step=1, description='Head:', value=0)
    widgets.interact(view_head, head_index=head_slider)
except Exception as e:
    print(f"Error creating interactive widget: {e}")